# 月度分时电量均摊
先将账单转录为 12 行月度分时电量表，列为 月份、尖、峰、平、谷。不存在的时段填 0。

In [ ]:
from load_common import export, hourly_frame, fill_meter_gaps, PERIODS
import numpy as np
import pandas as pd

YEAR = 2026
TARIFF = "inputs/tariff.xlsx"
INPUT = "inputs/monthly_energy.xlsx"
OUTPUT = "outputs/monthly_bills_8760.xlsx"
SHEET = 0


In [ ]:
frame = hourly_frame(YEAR, TARIFF)
raw = pd.read_excel(INPUT, sheet_name=SHEET).set_index("月份")[PERIODS]
if raw.index.has_duplicates or set(raw.index) != set(range(1, 13)):
    raise ValueError("月度电量表须含整数月份 1–12，每月一行")
bills = raw.apply(pd.to_numeric, errors="raise")
if not np.isfinite(bills.to_numpy()).all() or (bills < 0).any().any():
    raise ValueError("月度分时电量应为非负数，缺省时段填 0")
bills.columns.name = "电价类型"
energy = bills.stack().rename("账单电量_kWh")
counts = frame.groupby(["月份", "电价类型"]).size()
audit = energy.to_frame().join(counts.rename("小时数")).fillna({"小时数": 0})
if ((audit["小时数"] == 0) & (audit["账单电量_kWh"] > 0)).any():
    raise ValueError("账单有电量的时段在电价表中没有对应小时")
power = audit["账单电量_kWh"].div(audit["小时数"].replace(0, np.nan))
frame = frame.join(power.rename("负载"), on=["月份", "电价类型"])
frame["负载来源"] = "月度分时均摊"
audit = audit.join(frame.groupby(["月份", "电价类型"])["负载"].sum().rename("反算电量_kWh"))
audit["反算电量_kWh"] = audit["反算电量_kWh"].fillna(0)
audit["差额_kWh"] = audit["反算电量_kWh"] - audit["账单电量_kWh"]
if not np.allclose(audit["差额_kWh"], 0, atol=1e-6):
    raise ValueError("反算分时电量与账单不一致")
export(frame, OUTPUT, audit.reset_index())
